# California House Price Prediction

Full walkthrough: data cleaning -> feature engineering -> EDA -> model training -> evaluation.

## 1. Load & Inspect Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/housing.csv')
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 2. Data Cleaning

- Impute `total_bedrooms` missing values with the median, grouped by `ocean_proximity`
- Check for duplicates
- Flag the capped `median_house_value` values (known dataset artifact at $500,001)

In [ ]:
df_clean = df.copy()

# Duplicates
print('Duplicates:', df_clean.duplicated().sum())

# Impute missing total_bedrooms by ocean_proximity group median
df_clean['total_bedrooms'] = df_clean.groupby('ocean_proximity')['total_bedrooms'].transform(lambda x: x.fillna(x.median()))
print('Remaining nulls:', df_clean.isnull().sum().sum())

# Flag capped target values
n_capped = (df_clean['median_house_value'] >= 500001).sum()
print(f'Capped rows: {n_capped} ({n_capped/len(df_clean)*100:.2f}%)')

## 3. Feature Engineering

Add ratio features that are known to correlate well with house value, and one-hot encode `ocean_proximity`.

In [ ]:
df_feat = df_clean.copy()

df_feat['rooms_per_household'] = df_feat['total_rooms'] / df_feat['households']
df_feat['bedrooms_per_room'] = df_feat['total_bedrooms'] / df_feat['total_rooms']
df_feat['population_per_household'] = df_feat['population'] / df_feat['households']
df_feat['log_median_house_value'] = np.log1p(df_feat['median_house_value'])

df_feat = pd.get_dummies(df_feat, columns=['ocean_proximity'])
df_feat.head()

## 4. Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

df_clean.hist(bins=50, figsize=(14,10))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
sc = plt.scatter(df_clean['longitude'], df_clean['latitude'],
                  c=df_clean['median_house_value'], cmap='viridis',
                  alpha=0.4, s=df_clean['population']/100)
plt.colorbar(sc, label='Median House Value ($)')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.title('Geographic Distribution of House Values')
plt.show()

In [ ]:
numeric_df = df_feat.select_dtypes(include='number')
corr = numeric_df.corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

corr['median_house_value'].sort_values(ascending=False)

**Key insight:** `median_income` is by far the strongest predictor of house value. Geography (lat/long) and ocean proximity add meaningful secondary signal.

## 5. Model Training & Comparison

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X = df_feat.drop(columns=['median_house_value', 'log_median_house_value'])
y = df_feat['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'LinearRegression': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'RandomForest': RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    results.append({'model': name, 'rmse': rmse, 'mae': mae, 'r2': r2})
    print(f'{name:20s} RMSE={rmse:,.0f}  MAE={mae:,.0f}  R2={r2:.4f}')

results_df = pd.DataFrame(results).sort_values('rmse')
results_df

## 6. Feature Importance (Best Model)

In [ ]:
best_model = models['RandomForest']
importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,6))
importances.head(10).sort_values().plot(kind='barh')
plt.title('Top 10 Feature Importances')
plt.tight_layout()
plt.show()

importances.head(10)

## 7. Save Final Model

In [ ]:
import joblib
joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(list(X.columns), '../models/feature_columns.pkl')
print('Model saved.')

## 8. Conclusion

- Random Forest outperformed Linear Regression and Gradient Boosting on this dataset (R2 ~ 0.81 vs 0.60 / 0.78).
- `median_income` is the dominant driver of house value, followed by location-related features.
- Next steps: try XGBoost/LightGBM, add geographic clustering features, and deploy via the accompanying Streamlit app (`app.py`).